# 18 - Computer Vision Geometry

## What / Why / How

**What we are trying to do:** Understand pinhole camera projection, depth back-projection, and stereo depth.

**Why this matters:** Robot perception depends on turning pixels into geometry. Calibration errors here become bad grasps, maps, and navigation.

**How we will do it:** Project 3D points into image pixels, reconstruct them with depth, and recover depth from stereo disparity.

## Goal

Robotics vision starts with geometry.

You will implement:

- Pinhole camera projection
- Depth back-projection
- Stereo triangulation

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
fx, fy = 500.0, 500.0
cx, cy = 320.0, 240.0
K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

points_3d = np.array([
    [0.0, 0.0, 2.0],
    [0.2, 0.1, 2.5],
    [-0.3, 0.2, 3.0],
    [0.1, -0.2, 1.5],
])

def project(points):
    uvw = (K @ points.T).T
    return uvw[:, :2] / uvw[:, 2:3]

pixels = project(points_3d)
print("pixels:\n", pixels)

In [ ]:
def backproject(pixel, depth):
    u, v = pixel
    x = (u - cx) * depth / fx
    y = (v - cy) * depth / fy
    return np.array([x, y, depth])

reconstructed = np.array([backproject(p, d) for p, d in zip(pixels, points_3d[:, 2])])
print("reconstruction error:", np.linalg.norm(reconstructed - points_3d, axis=1))

In [ ]:
baseline = 0.12
left_u = pixels[:, 0]
right_u = left_u - fx * baseline / points_3d[:, 2]
disparity = left_u - right_u
stereo_depth = fx * baseline / disparity
print("true depth:", points_3d[:, 2])
print("stereo depth:", stereo_depth)

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 4))
    plt.scatter(pixels[:, 0], pixels[:, 1])
    plt.xlim(0, 640)
    plt.ylim(480, 0)
    plt.grid(True, alpha=0.3)
    plt.title("Projected image points")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add pixel noise and measure depth error.
2. Increase baseline and compare depth accuracy.
3. Explain why calibration is essential before robot vision is useful.